# 배달 시간 예측 실습 (Amazon Delivery Dataset)

Kaggle [Amazon Delivery Dataset](https://www.kaggle.com/datasets/sujalsuthar/amazon-delivery-dataset)으로
배달 소요 시간(분)을 예측하는 회귀 실습이에요.

**준비**: Kaggle에서 `amazon_delivery.csv`를 내려받아 저장소의 `data/` 폴더에 넣어 주세요.

**흐름**: 문제 정의 → 데이터 확인 → 정제·파생변수 → EDA → 분할 → 전처리 파이프라인 → 모델 비교 → 해석

## 0. 문제 정의 (Q1~Q5)

| 질문 | 답 |
|---|---|
| Q1. 입력(X)은? | 배달원 정보(`Agent_Age`, `Agent_Rating`), 가게·배달지 위치(→ 거리), 주문·픽업 시각, 환경(`Weather`, `Traffic`, `Area`), `Vehicle`, `Category` |
| Q2. 목표(Y)는? | `Delivery_Time` (분) |
| Q3. 분류? 회귀? | **회귀** — 연속적인 숫자를 예측 |
| Q4. 기계가 배울 것은? | 거리·교통·날씨·배달원 상태 등과 배달 시간 사이의 관계 f(X) ≈ Y |
| Q5. 예상 문제점은? | 결측치, 좌표 이상치, 시간 형식, 범주형 인코딩, 스케일 차이, 데이터 누수, 누락된 중요 요인 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

# 노트북을 저장소 루트나 notebooks/ 폴더 어디서 열어도 동작하게 경로를 찾는다
DATA_PATH = next(
    (p for p in [Path("data/amazon_delivery.csv"), Path("../data/amazon_delivery.csv")] if p.exists()),
    None,
)
assert DATA_PATH is not None, "data/amazon_delivery.csv 를 찾지 못했어요. Kaggle에서 내려받아 data/ 폴더에 넣어 주세요."
print("데이터 경로:", DATA_PATH)

## 1. 데이터 불러오기와 첫 확인

In [ ]:
df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# 결측치 개수와 비율
missing = df.isna().sum().to_frame("missing")
missing["ratio(%)"] = (missing["missing"] / len(df) * 100).round(2)
missing[missing["missing"] > 0]

In [ ]:
print("완전히 중복된 행:", df.duplicated().sum())
print("Order_ID 중복:", df["Order_ID"].duplicated().sum())
df.describe().T

> **확인 포인트**: 위 결과에서 결측이 있는 컬럼, 값 범위가 이상한 컬럼(나이, 평점, 좌표 등)을 먼저 메모해 두세요.
> 문자열 컬럼에 공백이나 `"NaN"` 같은 문자열이 섞여 있을 수 있어요.

In [ ]:
for c in ["Weather", "Traffic", "Vehicle", "Area", "Category"]:
    print(f"--- {c} ---")
    print(df[c].value_counts(dropna=False).to_string())
    print()

## 2. 목표 변수(Delivery_Time) 살펴보기

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Delivery_Time"], bins=50, kde=True, ax=axes[0])
axes[0].set_title("Delivery_Time distribution")
sns.boxplot(x=df["Delivery_Time"], ax=axes[1])
axes[1].set_title("Delivery_Time boxplot")
plt.tight_layout()
plt.show()

df["Delivery_Time"].describe()

## 3. 정제와 파생변수

- 문자열 컬럼: 앞뒤 공백 제거, `"NaN"`/빈 문자열을 진짜 결측으로 변환, 대소문자 통일
- 좌표: 0이면 결측으로 보고, 가게·배달지 좌표로 **하버사인 거리(km)** 계산
- 시간: 주문 시각, 요일, 주문→픽업 대기 시간(분) 파생

> 여기서 하는 정제는 행을 지우지 않고 값만 결측으로 바꿔요. 결측을 어떻게 채울지는
> 데이터 누수를 막기 위해 **분할 이후 파이프라인 안에서** 결정해요.

In [ ]:
data = df.drop_duplicates().copy()

# 1) 범주형 문자열 정리
cat_cols = ["Weather", "Traffic", "Vehicle", "Area", "Category"]
for c in cat_cols:
    s = data[c].astype("string").str.strip()
    s = s.mask(s.str.lower().isin(["nan", "none", ""]))
    s = s.str.title()
    data[c] = s.astype(object).where(s.notna(), np.nan)

# 2) 숫자형 변환 (변환 실패는 결측 처리)
for c in ["Agent_Age", "Agent_Rating", "Delivery_Time"]:
    data[c] = pd.to_numeric(data[c], errors="coerce")

# 3) 목표값이 없는 행은 학습에 쓸 수 없으니 제거
before = len(data)
data = data.dropna(subset=["Delivery_Time"])
print(f"목표값 결측으로 제거한 행: {before - len(data)}")

In [ ]:
# 4) 좌표 → 거리(km)
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


coord_cols = ["Store_Latitude", "Store_Longitude", "Drop_Latitude", "Drop_Longitude"]
zero_coord = (data[coord_cols].abs() < 1e-6).any(axis=1)
print("좌표에 0이 있는 행:", int(zero_coord.sum()))

data["Distance_km"] = haversine_km(
    data["Store_Latitude"], data["Store_Longitude"], data["Drop_Latitude"], data["Drop_Longitude"]
)
data.loc[zero_coord, "Distance_km"] = np.nan

# 비정상적으로 먼 거리는 좌표 오류일 가능성이 커요. 아래 분포를 보고 기준을 조정하세요.
MAX_DISTANCE_KM = 100
print(data["Distance_km"].describe())
too_far = data["Distance_km"] > MAX_DISTANCE_KM
print(f"{MAX_DISTANCE_KM}km 초과 행: {int(too_far.sum())}")
data.loc[too_far, "Distance_km"] = np.nan

In [ ]:
# 5) 시간 파생변수
def to_minutes(s):
    t = pd.to_datetime(s.astype("string").str.strip(), format="mixed", errors="coerce")
    return t.dt.hour * 60 + t.dt.minute


order_date = pd.to_datetime(data["Order_Date"], errors="coerce", format="mixed")
order_min = to_minutes(data["Order_Time"])
pickup_min = to_minutes(data["Pickup_Time"])

data["Order_Hour"] = order_min // 60
data["Day_Of_Week"] = order_date.dt.dayofweek  # 0=월 ... 6=일

wait = pickup_min - order_min
wait = wait.where(wait >= 0, wait + 1440)  # 자정을 넘긴 경우
data["Pickup_Wait_Min"] = wait.where(wait <= 180)  # 3시간 초과는 이상값으로 보고 결측 처리

data[["Distance_km", "Order_Hour", "Day_Of_Week", "Pickup_Wait_Min"]].describe().T

In [ ]:
num_features = ["Agent_Age", "Agent_Rating", "Distance_km", "Order_Hour", "Day_Of_Week", "Pickup_Wait_Min"]
cat_features = ["Weather", "Traffic", "Vehicle", "Area", "Category"]
target = "Delivery_Time"

# 이 단계 이후 결측 현황
data[num_features + cat_features].isna().mean().mul(100).round(2).rename("missing(%)").to_frame()

> **데이터 누수 점검**: 사용한 X는 모두 주문·픽업 시점에 알 수 있는 정보예요.
> `Delivery_Time`에서 만들어진 컬럼은 X에 넣지 않았어요. `Order_ID`와 원본 좌표·날짜 문자열도 제외했어요.

## 4. 탐색적 데이터 분석 (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, c in zip(axes.ravel(), cat_features):
    order = data.groupby(c)[target].median().sort_values().index
    sns.boxplot(data=data, x=c, y=target, order=order, ax=ax)
    ax.set_title(f"Delivery_Time by {c}")
    ax.tick_params(axis="x", rotation=30)
sns.scatterplot(data=data.sample(min(5000, len(data)), random_state=RANDOM_STATE),
                x="Distance_km", y=target, alpha=0.3, ax=axes[1, 2])
axes[1, 2].set_title("Distance_km vs Delivery_Time")
plt.tight_layout()
plt.show()

In [ ]:
# 숫자형 변수 상관관계
corr = data[num_features + [target]].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation (numeric features)")
plt.show()

corr[target].drop(target).sort_values(key=abs, ascending=False)

In [ ]:
# 주문 시간대별 평균 배달 시간
by_hour = data.groupby("Order_Hour")[target].agg(["mean", "count"])
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(by_hour.index, by_hour["mean"], marker="o")
ax.set_xlabel("Order_Hour")
ax.set_ylabel("mean Delivery_Time")
ax.set_title("Mean Delivery_Time by order hour")
plt.show()

> **생각해 보기**: 거리만으로 배달 시간이 잘 설명되나요? 교통·날씨·배달원 평점 같은 다른 변수가 더해지면 어떻게 달라질까요?

## 5. 학습/테스트 분할

전처리(결측 대체, 스케일링, 인코딩)의 통계는 **학습 데이터에서만** 계산해야 해요.
그래서 먼저 데이터를 나눈 뒤, 파이프라인 안에서 전처리를 학습시켜요.

In [ ]:
X = data[num_features + cat_features]
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("train:", X_train.shape, "test:", X_test.shape)

## 6. 전처리 파이프라인과 모델 비교

- 숫자형: 중앙값으로 결측 대체 → 표준화
- 범주형: 최빈값으로 결측 대체 → 원-핫 인코딩 (처음 보는 범주는 무시)
- 비교 모델: 평균 예측(기준선), 선형회귀, 랜덤포레스트, 히스토그램 기반 그래디언트 부스팅

In [ ]:
def make_preprocessor():
    numeric = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([("num", numeric, num_features), ("cat", categorical, cat_features)])


models = {
    "Baseline (mean)": DummyRegressor(strategy="mean"),
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}

fitted = {}
rows = []
for name, model in models.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    rows.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    })

results = pd.DataFrame(rows).set_index("model").sort_values("RMSE")
results.round(3)

> **읽는 법**: MAE는 평균적으로 몇 분 틀리는지, RMSE는 큰 오차에 더 민감한 지표, R²는 설명하는 분산의 비율이에요.
> 기준선(평균 예측)보다 얼마나 좋아졌는지가 핵심이에요.

In [ ]:
# 학습 데이터 5-겹 교차검증 (선형회귀, 부스팅만 — 랜덤포레스트는 시간이 오래 걸려 제외)
for name in ["LinearRegression", "HistGradientBoosting"]:
    pipe = Pipeline([("prep", make_preprocessor()), ("model", models[name])])
    scores = -cross_val_score(pipe, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
    print(f"{name:22s} CV MAE: {scores.mean():.3f} (+/- {scores.std():.3f})")

## 7. 결과 해석

In [ ]:
best_name = results.index[0]
best = fitted[best_name]
print("최고 성능 모델:", best_name)

pred = best.predict(X_test)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_test, pred, alpha=0.2, s=8)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
axes[0].plot(lims, lims, "r--")
axes[0].set_xlabel("Actual")
axes[0].set_ylabel("Predicted")
axes[0].set_title(f"Actual vs Predicted ({best_name})")
sns.histplot(y_test - pred, bins=50, kde=True, ax=axes[1])
axes[1].set_title("Residuals (Actual - Predicted)")
plt.tight_layout()
plt.show()

In [ ]:
# 변수 중요도: 원래 컬럼 단위로 순서를 섞어 성능이 얼마나 떨어지는지 측정 (permutation importance)
sample = X_test.sample(min(3000, len(X_test)), random_state=RANDOM_STATE)
perm = permutation_importance(
    best, sample, y_test.loc[sample.index],
    n_repeats=5, random_state=RANDOM_STATE, scoring="neg_mean_absolute_error",
)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()

imp.plot.barh(figsize=(7, 5))
plt.xlabel("MAE increase when shuffled")
plt.title(f"Permutation importance ({best_name})")
plt.tight_layout()
plt.show()

## 8. 정리 (직접 채워 보기)

- 가장 성능이 좋은 모델과 지표(MAE/RMSE/R²):
- 배달 시간에 가장 영향이 큰 변수 3가지:
- 기준선 대비 개선 폭:
- 모델이 잘 못 맞히는 구간(잔차 그래프 기준)과 그 이유로 짐작되는 것:
- 데이터에 없어서 아쉬운 변수:
- 다음에 해 볼 것 (하이퍼파라미터 튜닝, 이상치 처리 방식 비교, 분류 문제로 바꾸기 등):